## Dataset aur ML

In [10]:
from pathlib import Path
import pandas as pd

pred_path = Path(
    "C:/CN/AI-Adaptive-Routing/results/csv/route_cost_predictions.csv"
)

if pred_path.exists():
    predictions = pd.read_csv(pred_path)
    print("Prediction shape:", predictions.shape)
    print("Columns:", predictions.columns.tolist())
    display(predictions.head())
else:
    print("Prediction file not found.")

Prediction shape: (9, 14)
Columns: ['flow_id', 'source', 'destination', 'packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'traffic_level', 'packet_delivery_ratio_percent', 'route_cost', 'predicted_route_cost', 'prediction_error']


,flow_id,source,destination,packets_sent,packets_received,packets_lost,delay,throughput,jitter,traffic_level,packet_delivery_ratio_percent,route_cost,predicted_route_cost,prediction_error
0,1,10.1.2.2,10.1.9.2,9760,9760,0,0.019261,2.585600,4.385920e-05,low,100.0,0.299475,0.299475,0.000000
1,2,10.1.1.2,10.1.10.2,5493,5493,0,0.029590,2.714940,0.000000e+00,low,100.0,0.336913,0.299475,-0.037438
2,3,10.1.9.2,10.1.2.2,4882,4882,0,0.019022,0.119334,3.278020e-10,low,100.0,0.372506,0.372506,0.000000
3,1,10.1.2.2,10.1.9.2,38471,38471,0,0.019270,10.191500,4.366710e-05,medium,100.0,0.077082,0.077082,0.000000
4,2,10.1.1.2,10.1.10.2,21972,21972,0,0.029590,10.859000,0.000000e+00,medium,100.0,0.118361,0.118361,0.000000


## NS-3 ka executable locate

In [11]:
from pathlib import Path

roots = [
    Path("C:/ns3"),
    Path("C:/CN"),
    Path.home() / "Downloads",
]

for root in roots:
    if not root.exists():
        continue

    print("\nChecking:", root)

    for pattern in ["ns3", "ns3.exe", "build.ninja", "CMakeCache.txt"]:
        for item in list(root.rglob(pattern))[:5]:
            print(item)


Checking: C:\ns3
C:\ns3\ns-3.47\ns3
C:\ns3\ns-3.47\build\include\ns3
C:\ns3\ns-3.47\cmake-cache\build.ninja
C:\ns3\ns-3.47\cmake-cache\CMakeCache.txt

Checking: C:\CN
C:\CN\AI-Adaptive-Routing\ns3

Checking: C:\Users\azads\Downloads


In [13]:

from pathlib import Path

project = Path("C:/CN/AI-Adaptive-Routing")

required_files = [
    "ns3/topology.cc",
    "ns3/adaptive_topology.cc",
    "dataset/processed/flow_analysis.csv",
    "ml/route_cost_model.pkl",
    "results/csv/route_cost_predictions.csv",
]

for name in required_files:
    path = project / name
    print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")


ns3/topology.cc: FOUND
ns3/adaptive_topology.cc: FOUND
dataset/processed/flow_analysis.csv: FOUND
ml/route_cost_model.pkl: FOUND
results/csv/route_cost_predictions.csv: FOUND


In [12]:

from pathlib import Path

source = project / "ns3" / "adaptive_topology.cc"

if source.exists():
    code = source.read_text(errors="replace")

    for term in [
        "PopulateRoutingTables",
        "SetRoutingProtocol",
        "Ipv4StaticRouting",
        "SetDefaultRoute",
        "FlowMonitor",
        "network_data.csv",
    ]:
        print(f"{term}: {term in code}")

    print("\nSource length:", len(code), "characters")
else:
    print("adaptive_topology.cc not found")


PopulateRoutingTables: True
SetRoutingProtocol: False
Ipv4StaticRouting: False
SetDefaultRoute: False
FlowMonitor: True
network_data.csv: True

Source length: 12481 characters


## Implement AI Routing Engine 

In [16]:

from pathlib import Path
import joblib
import pandas as pd

PROJECT = Path("C:/CN/AI-Adaptive-Routing")
MODEL_PATH = PROJECT / "ml" / "route_cost_model.pkl"
PREDICTIONS_PATH = PROJECT / "results/csv/route_cost_predictions.csv"

saved = joblib.load(MODEL_PATH)

if isinstance(saved, dict):
    model = saved["model"]
    features = saved["features"]
    print("Model:", saved.get("model_name"))
    print("Features:", features)
else:
    model = saved
    features = None

if PREDICTIONS_PATH.exists():
    predictions = pd.read_csv(PREDICTIONS_PATH)
    print("\nPrediction rows:", len(predictions))
    display(predictions.head())
else:
    print("Prediction file not found.")


Model: Decision Tree
Features: ['packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'packet_delivery_ratio_percent']

Prediction rows: 9


,flow_id,source,destination,packets_sent,packets_received,packets_lost,delay,throughput,jitter,traffic_level,packet_delivery_ratio_percent,route_cost,predicted_route_cost,prediction_error
0,1,10.1.2.2,10.1.9.2,9760,9760,0,0.019261,2.585600,4.385920e-05,low,100.0,0.299475,0.299475,0.000000
1,2,10.1.1.2,10.1.10.2,5493,5493,0,0.029590,2.714940,0.000000e+00,low,100.0,0.336913,0.299475,-0.037438
2,3,10.1.9.2,10.1.2.2,4882,4882,0,0.019022,0.119334,3.278020e-10,low,100.0,0.372506,0.372506,0.000000
3,1,10.1.2.2,10.1.9.2,38471,38471,0,0.019270,10.191500,4.366710e-05,medium,100.0,0.077082,0.077082,0.000000
4,2,10.1.1.2,10.1.10.2,21972,21972,0,0.029590,10.859000,0.000000e+00,medium,100.0,0.118361,0.118361,0.000000


In [17]:
from pathlib import Path
import joblib
import pandas as pd

PROJECT = Path("C:/CN/AI-Adaptive-Routing")

saved = joblib.load(PROJECT / "ml/route_cost_model.pkl")

model = saved["model"] if isinstance(saved, dict) else saved
features = saved.get("features", []) if isinstance(saved, dict) else []

print("ML model loaded successfully")
print("Features:", features)

path = PROJECT / "results/csv/route_cost_predictions.csv"

if path.exists():
    predictions = pd.read_csv(path)
    print("Prediction rows:", len(predictions))
    display(predictions.head())
else:
    print("Prediction CSV not found")

ML model loaded successfully
Features: ['packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'packet_delivery_ratio_percent']
Prediction rows: 9


,flow_id,source,destination,packets_sent,packets_received,packets_lost,delay,throughput,jitter,traffic_level,packet_delivery_ratio_percent,route_cost,predicted_route_cost,prediction_error
0,1,10.1.2.2,10.1.9.2,9760,9760,0,0.019261,2.585600,4.385920e-05,low,100.0,0.299475,0.299475,0.000000
1,2,10.1.1.2,10.1.10.2,5493,5493,0,0.029590,2.714940,0.000000e+00,low,100.0,0.336913,0.299475,-0.037438
2,3,10.1.9.2,10.1.2.2,4882,4882,0,0.019022,0.119334,3.278020e-10,low,100.0,0.372506,0.372506,0.000000
3,1,10.1.2.2,10.1.9.2,38471,38471,0,0.019270,10.191500,4.366710e-05,medium,100.0,0.077082,0.077082,0.000000
4,2,10.1.1.2,10.1.10.2,21972,21972,0,0.029590,10.859000,0.000000e+00,medium,100.0,0.118361,0.118361,0.000000


In [18]:
from pathlib import Path
import joblib
import pandas as pd

PROJECT = Path("C:/CN/AI-Adaptive-Routing")

saved = joblib.load(PROJECT / "ml/route_cost_model.pkl")
model = saved["model"] if isinstance(saved, dict) else saved
features = saved.get("features", []) if isinstance(saved, dict) else []

data_path = PROJECT / "dataset/processed/flow_analysis.csv"
df = pd.read_csv(data_path)

print("Model loaded:", type(model).__name__)
print("Dataset rows:", len(df))
print("Required features:", features)
print("Available columns:", df.columns.tolist())

missing = [c for c in features if c not in df.columns]

if missing:
    print("\nCannot predict yet. Missing model features:", missing)
else:
    X = df[features].apply(pd.to_numeric, errors="coerce")
    valid = X.notna().all(axis=1)

    if not valid.any():
        print("No rows contain all required numeric features.")
    else:
        result = df.loc[valid].copy()
        result["predicted_route_cost"] = model.predict(X.loc[valid])

        output = PROJECT / "results/csv/ai_routing_engine_predictions.csv"
        output.parent.mkdir(parents=True, exist_ok=True)
        result.to_csv(output, index=False)

        print("\nPredictions saved to:", output)
        display(result.head())

Model loaded: DecisionTreeRegressor
Dataset rows: 9
Required features: ['packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'packet_delivery_ratio_percent']
Available columns: ['flow_id', 'source', 'destination', 'packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'traffic_level', 'packet_delivery_ratio_percent', 'route_cost']

Predictions saved to: C:\CN\AI-Adaptive-Routing\results\csv\ai_routing_engine_predictions.csv


,flow_id,source,destination,packets_sent,packets_received,packets_lost,delay,throughput,jitter,traffic_level,packet_delivery_ratio_percent,route_cost,predicted_route_cost
0,1,10.1.2.2,10.1.9.2,9760,9760,0,0.019261,2.585600,4.385920e-05,low,100.0,0.299475,0.299475
1,2,10.1.1.2,10.1.10.2,5493,5493,0,0.029590,2.714940,0.000000e+00,low,100.0,0.336913,0.299475
2,3,10.1.9.2,10.1.2.2,4882,4882,0,0.019022,0.119334,3.278020e-10,low,100.0,0.372506,0.372506
3,1,10.1.2.2,10.1.9.2,38471,38471,0,0.019270,10.191500,4.366710e-05,medium,100.0,0.077082,0.077082
4,2,10.1.1.2,10.1.10.2,21972,21972,0,0.029590,10.859000,0.000000e+00,medium,100.0,0.118361,0.118361
